In [0]:
# =====================================================================================
# Notebook: 99_validation/02_silver_checker.py
# Purpose : Validate Silver tables (freshness + sanity)
# =====================================================================================

from pyspark.sql import functions as F

SILVER_DIM_TEAM = "workspace.nba_silver.dim_team"
SILVER_FACT_GAME = "workspace.nba_silver.fact_game"

dim_team = spark.table(SILVER_DIM_TEAM)
fact_game = spark.table(SILVER_FACT_GAME)

print("=== Row counts ===")
print("dim_team:", dim_team.count())
print("fact_game:", fact_game.count())

print("\n=== Latest timestamps (freshness) ===")
fact_game.select(
    F.max("source_dt").alias("max_source_dt"),
    F.max("ingested_at").alias("max_ingested_at"),
    F.max("start_time_utc").alias("max_start_time_utc"),
    F.max("game_date_utc").alias("max_game_date_utc"),
).show(truncate=False)

print("\n=== Recent games (by start_time_utc) ===")
fact_game.select(
    "game_id_bdl","season_start_year","game_date_utc","start_time_utc",
    "status","home_score","visitor_score","home_team_id_bdl","visitor_team_id_bdl","source_dt"
).orderBy(F.col("start_time_utc").desc()).show(25, truncate=False)

print("\n=== Status distribution (top) ===")
fact_game.groupBy("status").count().orderBy(F.col("count").desc()).show(50, truncate=False)

print("\n=== Team join coverage (should be near 100%) ===")
joined = (
    fact_game.alias("g")
    .join(dim_team.alias("ht"), F.col("g.home_team_id_bdl") == F.col("ht.team_id_bdl"), "left")
    .join(dim_team.alias("vt"), F.col("g.visitor_team_id_bdl") == F.col("vt.team_id_bdl"), "left")
)

joined.select(
    F.sum(F.when(F.col("ht.team_id_bdl").isNull(), 1).otherwise(0)).alias("missing_home_team"),
    F.sum(F.when(F.col("vt.team_id_bdl").isNull(), 1).otherwise(0)).alias("missing_visitor_team"),
    F.count("*").alias("total_games"),
).show(truncate=False)

print("\n=== If missing teams exist, sample them ===")
joined.filter(F.col("ht.team_id_bdl").isNull() | F.col("vt.team_id_bdl").isNull()) \
      .select("g.game_id_bdl","g.home_team_id_bdl","g.visitor_team_id_bdl","g.start_time_utc","g.source_dt") \
      .orderBy(F.col("g.start_time_utc").desc()) \
      .show(20, truncate=False)
